# TaskVine smoke test

TaskVine manager in the notebook (port 8786, TLS) with workers submitted to HTCondor by `coffea_casa.CoffeaCasaVineWorkers`:

- **A:** workers connect through the facility route, and files move between workers.
- **B:** a worker reads CMS data through XCache with the user's token.

Needs a fresh kernel with no Dask cluster running (both use port 8786). Any failed check raises, so an automated run fails. A failure leaves the workers to exit after `idle_timeout`.

In [ ]:
import time
import ndcctools.taskvine as vine
from coffea_casa import CoffeaCasaVineWorkers, vine_manager

m = vine_manager()
workers = CoffeaCasaVineWorkers(m, cores=1, memory_mb=2000, disk_mb=4000, idle_timeout=300)
print("manager:", workers.manager_address)
print("condor cluster:", workers.submit(2))


def run(tasks, timeout=900):
    """Submit `tasks`, wait for all of them, raise if any fails or times out."""
    pending = {m.submit(t): t for t in tasks}
    start = time.time()
    while pending and time.time() - start < timeout:
        t = m.wait(10)
        if t is not None and t.id in pending:
            del pending[t.id]
            out = (t.output or "").strip().replace("\n", " | ")
            print(f"[{time.time() - start:5.0f}s] task {t.id}: ok={t.successful()} "
                  f"result={t.result} exit={t.exit_code} output={out!r}")
    assert not pending, f"tasks not finished after {timeout}s: {sorted(pending)}"
    failed = [t.id for t in tasks if not t.successful()]
    assert not failed, f"tasks failed: {failed}"

## A: workers connect, files move between workers

One task writes a file on its worker; 8 single-core tasks read it. With two one-core workers, some run on the other worker, which fetches the file from the first one.

In [ ]:
f = m.declare_temp()
producer = vine.Task("python -c 'print(41 + 1)' > out.txt")
producer.add_output(f, "out.txt")
consumers = []
for _ in range(8):
    c = vine.Task("cat in.txt; hostname")
    c.add_input(f, "in.txt")
    consumers.append(c)

run([producer] + consumers)
assert all(c.output.split()[0] == "42" for c in consumers), "unexpected consumer output"
hosts = sorted({c.output.split()[-1] for c in consumers})
print("hosts:", hosts)
assert len(hosts) == 2, f"expected tasks on both workers, got {hosts}"

## B: CMS data through XCache from a worker

In [ ]:
url = "root://xcache//store/data/Run2018A/DoubleMuon/NANOAOD/02Apr2020-v1/30000/0555868D-6B32-D249-9ED1-6B9A6AABDAF7.root"
x = vine.Task(f"""python -c "import uproot; print(uproot.open('{url}')['Events'].num_entries)" """)
run([x], timeout=300)
n_events = int(x.output.strip().split()[-1])
print("events:", n_events)
assert n_events > 0

## Clean up

In [ ]:
workers.close()
print("TaskVine smoke test passed")